# CCTV Safety — Stage 1 YOLOv8s Training

This notebook describes the fresh YOLOv8s experiment that has since been run for comparison with the Stage 1 YOLOv8n baseline; current results are in `docs/project_status.md`. It uses the provisional six-class dataset and does not resume or reuse YOLOv8n weights.

Run cells in order. The training gate is `False` by default. Before enabling it, attach the private Stage 1 dataset, enable Kaggle T4 x2, and ensure Kaggle Internet is on (or attach `yolov8s.pt` as an input). The training cell uses 100 epochs, batch 16, image size 640, and 2 workers.

The run writes to `/kaggle/working/stage1_runs/stage1_yolov8s/`, separate from YOLOv8n. Test evaluation uses the held-out test split only after training. Dataset is provisional; results are experimental, not production approval.

In [ ]:
from pathlib import Path
import csv, json, shutil, zipfile

INPUT_ROOT = Path('/kaggle/input')
TEMP_ROOT = Path('/kaggle/temp')
WORK_ROOT = Path('/kaggle/working')
DATA_ROOT = None  # resolved to Kaggle's read-only input or a temp extraction path below
DATA_YAML = WORK_ROOT / 'stage1_yolov8s_data.yaml'  # writable; never write into /kaggle/input
RUN_ROOT = WORK_ROOT / 'stage1_runs' / 'stage1_yolov8s'
RUN_NAME = 'yolov8s_run2'  # Fresh output name; avoids the prior incomplete attempt.
MODEL_RUN_DIR = RUN_ROOT / RUN_NAME
DEVICE = '0,1'
EXPECTED_NAMES = ['person', 'helmet', 'vest', 'fall', 'fire', 'smoke']
DATA_ZIP_NAME = 'stage1_unified_colab.zip'

manifests = [p for p in INPUT_ROOT.rglob('dataset_manifest.json') if p.is_file()]
archives = [p for p in INPUT_ROOT.rglob(DATA_ZIP_NAME) if p.is_file()]
if len(manifests) > 1 or len(archives) > 1:
    raise RuntimeError('Attach one copy of the Stage 1 dataset only.')
if not manifests and not archives:
    raise FileNotFoundError('Attach the private Stage 1 dataset ZIP or its expanded files.')
DATA_SOURCE_ROOT = manifests[0].parent if manifests else None
DATA_ARCHIVE = archives[0] if archives else None
print('Dataset source:', DATA_SOURCE_ROOT or DATA_ARCHIVE)
print('Training output:', RUN_ROOT)

In [ ]:
def safe_extract(zip_path, destination):
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            relative = Path(info.filename.replace('\\', '/'))
            target = (destination / relative).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError('Unsafe ZIP member path: {}'.format(info.filename))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)

if DATA_SOURCE_ROOT is not None:
    # Kaggle already expanded the dataset. Read it in place; do not duplicate thousands of files.
    DATA_ROOT = DATA_SOURCE_ROOT
else:
    # Extract only when Kaggle exposes the uploaded dataset as a ZIP archive.
    DATA_ROOT = TEMP_ROOT / 'stage1_unified_yolov8s'
required = [DATA_ROOT / 'dataset_manifest.json', DATA_ROOT / 'data.yaml']
required += [DATA_ROOT / 'metadata' / '{}.csv'.format(s) for s in ('train', 'val', 'test')]
if DATA_SOURCE_ROOT is None and not all(p.is_file() for p in required):
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    safe_extract(DATA_ARCHIVE, DATA_ROOT)
assert all(p.is_file() for p in required), 'Dataset extraction/copy is incomplete.'
manifest = json.loads((DATA_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
assert manifest['class_names'] == EXPECTED_NAMES, manifest.get('class_names')
assert int(manifest['total_images']) == 23169, manifest.get('total_images')
data_text = "path: {}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 6\nnames: ['person', 'helmet', 'vest', 'fall', 'fire', 'smoke']\n".format(DATA_ROOT.resolve().as_posix())
DATA_YAML.write_text(data_text, encoding='utf-8')
print('Dataset staged:', manifest['total_images'], 'images;', manifest['total_instances'], 'boxes')
print('Provisional:', manifest.get('provisional'), '| validation_only:', manifest.get('validation_only'))
print(DATA_YAML.read_text(encoding='utf-8'))

In [ ]:
# Full split, image/label, schema, metadata, and group leakage preflight.
splits = ('train', 'val', 'test')
groups, image_counts = {}, {}
class_counts = {name: 0 for name in EXPECTED_NAMES}
suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in splits:
    images = [p for p in (DATA_ROOT / 'images' / split).glob('*') if p.is_file() and p.suffix.lower() in suffixes]
    labels = DATA_ROOT / 'labels' / split
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels.glob('*.txt')}
    assert image_stems == label_stems, '{} image/label mismatch'.format(split)
    image_counts[split] = len(images)
    with (DATA_ROOT / 'metadata' / '{}.csv'.format(split)).open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == len(images), '{} metadata count mismatch'.format(split)
    assert {r['image'] for r in rows} == {p.name for p in images}, '{} metadata filenames mismatch'.format(split)
    assert all(r['group_id'].strip() for r in rows), '{} has empty group_id'.format(split)
    groups[split] = {r['group_id'] for r in rows}
    for image in images:
        for line_no, line in enumerate((labels / '{}.txt'.format(image.stem)).read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            assert len(parts) == 5, '{}:{} malformed YOLO label'.format(image.name, line_no)
            cid = int(parts[0])
            assert 0 <= cid < 6, '{}:{} invalid class id'.format(image.name, line_no)
            class_counts[EXPECTED_NAMES[cid]] += 1
            x, y, w, h = map(float, parts[1:])
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, '{}:{} invalid normalized box'.format(image.name, line_no)
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not (groups[a] & groups[b]), 'Group leakage between {} and {}'.format(a, b)
assert image_counts == manifest['images_by_split'], (image_counts, manifest['images_by_split'])
assert class_counts == manifest['instances'], (class_counts, manifest['instances'])
print('DATA PREFLIGHT PASS')
print('Images by split:', image_counts)
print('Class instances:', class_counts)

## GPU and library preflight

Select **GPU T4 x2** in Kaggle settings. Run the install cell, then the GPU/library preflight. These cells do not start training. Ultralytics is pinned to 8.4.163 to match the YOLOv8n experiment. If `yolov8s.pt` is not attached as a private input, Kaggle Internet must be enabled so Ultralytics can fetch the official pretrained checkpoint.

In [ ]:
%pip install -q --no-deps ultralytics==8.4.163

In [ ]:
!nvidia-smi
import importlib.metadata, torch
from ultralytics import YOLO
version = importlib.metadata.version('ultralytics')
print('Ultralytics:', version, '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert version == '8.4.163', 'Expected Ultralytics 8.4.163; resolve version before training.'
assert torch.cuda.is_available(), 'Stop: enable a Kaggle GPU accelerator first.'
assert torch.cuda.device_count() >= 2, 'DEVICE requests T4 x2; enable both GPUs or change DEVICE to 0.'
for index in range(torch.cuda.device_count()):
    print('GPU {}:'.format(index), torch.cuda.get_device_name(index))

## Training gate — disabled by default

When ready to spend Kaggle GPU quota, change `TRAINING_APPROVED` to `True` and run this cell once. This starts a fresh YOLOv8s run from pretrained COCO weights; it does not resume YOLOv8n. Checkpoints are saved in a distinct output directory.

In [ ]:
TRAINING_APPROVED = False  # Change to True only when you choose to start YOLOv8s training.
if not TRAINING_APPROVED:
    print('Training remains disabled.')
else:
    if MODEL_RUN_DIR.exists():
        raise FileExistsError('Output already exists; use a new run name rather than overwrite it.')
    pretrained_inputs = [p for p in INPUT_ROOT.rglob('yolov8s.pt') if p.is_file()]
    if len(pretrained_inputs) > 1:
        raise RuntimeError('Attach no more than one yolov8s.pt input.')
    model_source = str(pretrained_inputs[0]) if pretrained_inputs else 'yolov8s.pt'
    print('Starting fresh from pretrained weights:', model_source)
    model = YOLO(model_source)
    model.train(
        data=str(DATA_YAML), epochs=100, batch=16, imgsz=640, workers=2,
        device=DEVICE, project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False,
        pretrained=True, resume=False, save=True, save_period=10, plots=True,
        seed=42, deterministic=True, patience=30,
    )
    (MODEL_RUN_DIR / 'TRAINING_COMPLETE').write_text('YOLOv8s training completed.', encoding='utf-8')
    print('Training output:', MODEL_RUN_DIR)

## Held-out test evaluation

Run after training completes. Uses `best.pt` and the held-out test split once. Save the resulting `stage1_runs/stage1_yolov8s/` output as a Kaggle version before ending the session.

In [ ]:
from ultralytics import YOLO
import json
best_path = MODEL_RUN_DIR / 'weights' / 'best.pt'
assert best_path.is_file(), 'Missing best.pt; complete training first or attach this notebook output.'
best_model = YOLO(str(best_path))
test_metrics = best_model.val(
    data=str(DATA_YAML), split='test', device='0', plots=True,
    project=str(RUN_ROOT), name='yolov8s_test', exist_ok=True,
)
box = test_metrics.box
with (MODEL_RUN_DIR / 'results.csv').open(encoding='utf-8', newline='') as f:
    train_rows = list(csv.DictReader(f))
trained_epochs = int(float(train_rows[-1]['epoch'])) if train_rows else None
per_class = {}
maps50 = list(box.ap50)
maps50_95 = list(box.maps)
for index, class_name in enumerate(EXPECTED_NAMES):
    per_class[class_name] = {'map50': float(maps50[index]), 'map50_95': float(maps50_95[index])}
results = {
    'model': 'yolov8s', 'trained_epochs': trained_epochs,
    'precision': float(box.mp), 'recall': float(box.mr),
    'map50': float(box.map50), 'map50_95': float(box.map),
    'per_class': per_class, 'class_names': EXPECTED_NAMES,
    'dataset_manifest': str(DATA_ROOT / 'dataset_manifest.json'),
    'provisional_dataset': bool(manifest.get('provisional', True)),
}
(RUN_ROOT / 'yolov8s_test_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print(json.dumps(results, indent=2))

## Preserve outputs

Use **Save Version → Quick Save → Save Output: Always** after the run. Confirm the output includes `stage1_runs/stage1_yolov8s/yolov8s/weights/best.pt`, `last.pt`, `results.csv`, `args.yaml`, and the test metrics/plots under `stage1_yolov8s/`. The dataset remains under `/kaggle/temp` and is not duplicated into saved outputs.